# M7 · 毕业挑战：提交有证据的优化 PR

**核心问题：** 一个优化 PR 至少需要哪些可复核证据？从官方数据生成学生分析记录，明确标为 `REFERENCE_ANALYSIS_NOT_NEW_GPU_RUN`，规划公平 A/B、正确性与资源门禁。默认不写文件。

**学习闭环：** 先预测 → 检查固定证据 → 独立计算 → 批判性解释 → 设计受控实验。

对应理论课：[打开 course/M07_student_challenge/CHALLENGES.zh-CN.md](../course/M07_student_challenge/CHALLENGES.zh-CN.md)。切换语言：[English](./08_capstone_and_pr.ipynb)。

**默认模式：** 纯 CPU、无需联网、只读仓库证据。不会自动启动 GPU。


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root (see notebooks/README).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab import notebook_support as ns
import statistics
print("Course repository:", ROOT)
print("Mode: OFFLINE / committed reference evidence; NO GPU work is launched by default.")


## 检查真实数据并独立计算

按照顺序执行；每个比例都要解释分母与适用范围。

In [ ]:
rows = ns.summary_rows()
b0, f1 = rows["B0"], rows["F1"]
evidence_record = {
    "origin": "committed reference evidence, not a new physical run",
    "status": "REFERENCE_ANALYSIS_NOT_NEW_GPU_RUN",
    "hypothesis": "attention backend may improve same-workload latency",
    "baseline": {"experiment": "B0", "warm_median_s": b0["warm_total_median_s"], "n": b0["n_measure"]},
    "candidate": {"experiment": "F1", "warm_median_s": f1["warm_total_median_s"], "n": f1["n_measure"]},
    "derived": {"speedup_x": round(b0["warm_total_median_s"] / f1["warm_total_median_s"], 4)},
    "source_paths": ["evidence/results/SUMMARY.json", "lab/reference_runs.json"],
    "limitations": ["single W7900-class host", "warm reference population", "student has not run a new A/B"],
    "quality_gate": "requires a separate review for the claimed deployment",
}
import json
print(json.dumps(evidence_record, indent=2, ensure_ascii=False))
assert evidence_record["status"] != "MEASURED_NEW_GPU_RUN"


In [ ]:
WRITE_STUDENT_RECORD = False  # Explicit opt-in only, never evidence/
if WRITE_STUDENT_RECORD:
    path = ns.student_record(evidence_record, "student-reference-analysis", confirmed=True, root=ROOT)
    print("Wrote student-only artifact:", path)
else:
    print("No file written. Set WRITE_STUDENT_RECORD=True to save under runs/notebook-student/.")


## 学生思考 / 自检

提出一个可以被证伪的优化假设及公平 A/B 对照：列出模型身份、源码 SHA、负载、中位数与方差、显存/内存、画质状态、负结果和复现命令。将证据提交为 Draft PR，不能把引用数据重算宣传成新真机测试。

**证据规范：** 用自己的语言完成回答。本 Notebook 重算的是已提交记录，不是你的 GPU 实测。如果自愿运行 smoke，结果必须放在 `runs/`，不能宣传为正式全量统计。
